In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
# LOAD STOCK DATA FROM CSV FILE

# Configuration - Change these variables as needed
START_DATE = '2017-01-01'
END_DATE = '2024-06-21'

# Load data from CSV
DATA_PATH = r"data/QQQ.csv"
stock_data = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)

# Filter by date range
stock_data = stock_data.loc[START_DATE:END_DATE]

if not stock_data.empty:
    print(f"Successfully loaded {len(stock_data)} records")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to load data from {DATA_PATH}")

# Display the data
stock_data

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def select_close_series(df):
    """Select close price column, handling both 'Close' and 'close' column names"""
    if isinstance(df.columns, pd.MultiIndex):
        # Handle MultiIndex columns
        cols = [c for c in df.columns if 'Close' in str(c) or 'close' in str(c)]
        if not cols:
            raise KeyError("Close/close column not found")
        s = df[cols[0]]
    else:
        # Handle regular columns - try 'Close' first, then 'close'
        if 'Close' in df.columns:
            s = df['Close']
        elif 'close' in df.columns:
            s = df['close']
        else:
            raise KeyError("Close/close column not found")
    return s.astype(float).squeeze()

close = select_close_series(stock_data)
close.name = 'price'

# Simple train/validation split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} → {train_close.index[-1].date()} | "
      f"val={val_close.index[0].date()} → {val_close.index[-1].date()}")

SIMPLE MA CROSSOVER GRID SEARCH
----------------------------------------------

**Strategy Logic**: Buy when price crosses above MA. Sell when price crosses below MA.

---

In [ ]:
# Simple MA Parameters

ma_periods = list(range(10, 210, 5))

print(f"Testing {len(ma_periods)} MA periods: {ma_periods[0]} to {ma_periods[-1]}")

In [ ]:
grid_search_results = []
FREQ = '1D'

In [ ]:
# SIMPLE MA GRID SEARCH

for ma_period in tqdm(ma_periods, desc="Testing MA periods"):
    try:
        ma = train_close.rolling(ma_period).mean()
        
        entries_raw = (train_close > ma) & (train_close.shift(1) <= ma.shift(1))
        exits_raw = (train_close < ma) & (train_close.shift(1) >= ma.shift(1))
        entries = entries_raw.shift(1).fillna(False)
        exits = exits_raw.shift(1).fillna(False)
        
        pf = vbt.Portfolio.from_signals(
            close=train_close.to_numpy(dtype=float),
            entries=entries.to_numpy(dtype=bool),
            exits=exits.to_numpy(dtype=bool),
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq=FREQ
        )
        
        sharpe = float(pf.sharpe_ratio(freq=FREQ))
        if not np.isnan(sharpe) and not np.isinf(sharpe):
            grid_search_results.append({
                'ma_period': ma_period,
                'sharpe_ratio': sharpe,
                'total_return': float(pf.total_return()),
                'max_drawdown': float(pf.max_drawdown())
            })
    except:
        pass

results_df = pd.DataFrame(grid_search_results)
print(f"Grid search complete: {len(results_df)} valid results")

In [ ]:
# Analysis cell
pass

In [ ]:
# Analysis cell
pass

In [ ]:
# Analysis cell
pass

In [ ]:
# Analysis cell
pass

In [ ]:
# Analysis cell
pass

In [ ]:
# Analysis cell
pass